# Phase 2: Sentinel-1 SAR Preprocessing & Seasonal Composite Dataset (2017-2026)

Mục tiêu của notebook này là xây dựng bộ dữ liệu Sentinel-1 SAR sạch, nhất quán và chuẩn nghiên cứu khoa học (paper-quality) cho khu vực Sông Hồng (Hà Nội) giai đoạn 2017–2026. Bộ dữ liệu được tiền xử lý thống nhất theo mùa (mùa mưa và mùa khô) để phục vụ cho các tác vụ phân loại mặt nước và bãi bồi ở các tuần tiếp theo.

## 1. Khởi tạo Google Earth Engine (GEE)

In [ ]:
import sys
import os

# Bổ sung thư mục gốc của project vào sys.path để import các module trong src
sys.path.insert(0, os.getcwd())

import ee
from src.config import GEE_PROJECT, START_YEAR, END_YEAR
from src.aoi import get_aoi_geometry

# Khởi tạo Earth Engine
try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"Earth Engine initialized successfully for project: {GEE_PROJECT}")
except Exception as e:
    print(f"GEE initialization failed: {e}. Running ee.Authenticate()...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

## 2. Thử nghiệm trên năm 2024 (Prototype Mode)

Trước khi chạy hàng loạt cho toàn bộ chuỗi thời gian 10 năm, chúng ta chạy thử nghiệm trên năm 2024 để đánh giá chất lượng của bộ lọc đốm thích ứng hướng **Refined Lee** và kiểm soát chất lượng (QC) bằng biểu đồ phân bố và bản đồ đối chiếu trực quan với Sentinel-2.

In [ ]:
from scripts.run_pipeline import run_prototype

# Chạy prototype cho năm 2024
run_prototype(2024)

### Kiểm soát Chất lượng (QC) Kết quả năm 2024

Sau khi chạy xong, hãy kiểm tra các tệp kết quả trong thư mục `outputs/`:
1. **Báo cáo chất lượng (JSON)**: `outputs/s1_dataset_metadata.json` chứa thông tin thống kê số lượng ảnh thô, giá trị phân tách VV tại vùng nước tham chiếu (bảo đảm $VV_{water} \le -15\text{ dB}$) và vùng đất cát tham chiếu (bảo đảm $VV_{land} \ge -10\text{ dB}$).
2. **Biểu đồ phân bố (Histogram)**: Xem các tệp `outputs/histogram_2024_dry.png` và `outputs/histogram_2024_wet.png` để đánh giá tính lưỡng cực (bimodal distribution) đặc trưng giúp phân tách rõ ranh giới nước/đất.
3. **Bản đồ tương tác đối chiếu (Folium Map)**: Mở các tệp HTML `outputs/comparison_2024_dry.html` và `outputs/comparison_2024_wet.html` trên trình duyệt để so sánh lớp ảnh SAR VV composite đã lọc nhiễu với ảnh vệ tinh quang học Sentinel-2 RGB không mây.

## 3. Chạy hàng loạt xuất dữ liệu sang GEE Asset (Production Mode)

Khi quá trình prototype năm 2024 được xác thực đạt chuẩn chất lượng khoa học, chúng ta tiến hành submit các task xuất dữ liệu hàng loạt cho toàn bộ các mùa từ **2017 đến 2026** lên Google Earth Engine để lưu thành các Asset phục vụ mô hình hóa.

In [ ]:
from scripts.run_pipeline import run_production

# Gửi toàn bộ các task xuất dữ liệu mùa khô và mùa mưa của các năm 2017-2026 lên GEE
submitted_tasks = run_production(start_year=2017, end_year=2026)

### Cách kiểm tra trạng thái Task trên GEE

Do quá trình tính toán Refined Lee và gộp ảnh trên chuỗi thời gian dài rất nặng, Earth Engine sẽ xử lý bất đồng bộ (asynchronously) ở chế độ nền trên hạ tầng cloud. Bạn có thể theo dõi tiến độ các task bằng cách:
1. Truy cập trang quản lý task trực quan: [Google Earth Engine Code Editor Tasks](https://code.earthengine.google.com/tasks)
2. Hoặc chạy dòng lệnh sau trong terminal để kiểm tra trực tiếp:
   ```bash
   earthengine task list
   ```
Sau khi các task chuyển sang màu xanh lá (COMPLETED), dữ liệu các composite seasonal đã sẵn sàng trong tab **Assets** của GEE để sử dụng.